In [2]:
# %% [markdown]
# BEACON — Preprocessing Steps 1-4
# Runs on the per-category Parquet files produced by beacon_convert_safe.py
# (net_<Category>.parquet, mem_<Category>.parquet in PROCESSED_DIR).
# Each step processes ONE category at a time to stay memory-safe.

# %%
import os
import polars as pl

# Windows-specific fix: pl.read_parquet() memory-maps the file by default,
# which keeps an OS-level handle open. Writing back to that same path then
# fails with "The requested operation cannot be performed on a file with a
# user-mapped section open" (OSError 1224). Two changes fix this:
#   1. Read with memory_map=False so no mapping is held.
#   2. Write to a temp file, then atomically replace the original —
#      also protects against a half-written file if the script crashes mid-write.
def read_parquet_safe(path: str) -> pl.DataFrame:
    return pl.read_parquet(path, memory_map=False)

def write_parquet_safe(df: pl.DataFrame, path: str) -> None:
    tmp_path = path + ".tmp"
    df.write_parquet(tmp_path)
    os.replace(tmp_path, path)  # atomic on both Windows and POSIX

PROCESSED_DIR = r"D:\Malware Dataset\processed"
CLEAN_DIR = r"D:\Malware Dataset\processed_clean"
os.makedirs(CLEAN_DIR, exist_ok=True)

CATEGORIES = ["Backdoor", "Benign", "Exploit", "HackTool", "Hoax",
              "Rootkit", "Trojan", "Virus", "Worm"]

# Known label inconsistency found during EDA: the Benign network category's
# internal label column uses "Zbenign" while everywhere else uses "Benign".
# Add any other variants you find here.
LABEL_FIXES = {
    "Zbenign": "Benign",
}

# %%
# ============================================================
# STEP 1: Remove exact duplicate rows — BEFORE any train/test split
# ============================================================
# Rationale (de Vargas et al., Sect. 2): deduping after a split can leave
# a duplicate pair split across train and test, letting the model be
# "evaluated" on a row it already memorized — inflating reported accuracy
# without real generalization.

def dedup_category(prefix: str, cat: str) -> dict:
    """Load one category's parquet, drop exact duplicate rows, save cleaned
    version. Returns a dict of before/after counts for reporting."""
    in_path = os.path.join(PROCESSED_DIR, f"{prefix}_{cat}.parquet")
    if not os.path.exists(in_path):
        return {"category": cat, "found": False}

    df = read_parquet_safe(in_path)
    before = df.height
    df = df.unique(keep="first")
    after = df.height
    removed = before - after

    out_path = os.path.join(CLEAN_DIR, f"{prefix}_{cat}_dedup.parquet")
    write_parquet_safe(df, out_path)

    return {
        "category": cat, "found": True,
        "before": before, "after": after, "removed": removed,
    }

print("=" * 60)
print("NETWORK — duplicate removal (compare against your known EDA counts)")
print("=" * 60)
net_dedup_report = []
for cat in CATEGORIES:
    result = dedup_category("net", cat)
    net_dedup_report.append(result)
    if result["found"]:
        print(f"  {cat}: removed {result['removed']} duplicates "
              f"({result['before']} -> {result['after']})")
    else:
        print(f"  {cat}: parquet not found — run beacon_convert_safe.py first")

# %%
print("=" * 60)
print("MEMORY — duplicate removal (not previously checked — new this run)")
print("=" * 60)
mem_dedup_report = []
for cat in CATEGORIES:
    result = dedup_category("mem", cat)
    mem_dedup_report.append(result)
    if result["found"]:
        print(f"  {cat}: removed {result['removed']} duplicates "
              f"({result['before']} -> {result['after']})")
    else:
        print(f"  {cat}: parquet not found — run beacon_convert_safe.py first")

# %%
# ============================================================
# STEP 2: Drop fully-empty / non-informative columns (Memory side)
# ============================================================
# info.winBuild is null in 100% of rows across every category — no
# imputation can recover information that was never captured. Drop it
# outright rather than filling it with a fabricated value.

COLUMNS_TO_DROP_MEMORY = ["info.winBuild"]

def drop_empty_columns(cat: str) -> None:
    in_path = os.path.join(CLEAN_DIR, f"mem_{cat}_dedup.parquet")
    if not os.path.exists(in_path):
        print(f"  {cat}: not found, skipping")
        return

    df = read_parquet_safe(in_path)
    existing_cols_to_drop = [c for c in COLUMNS_TO_DROP_MEMORY if c in df.columns]
    if existing_cols_to_drop:
        df = df.drop(existing_cols_to_drop)
        print(f"  {cat}: dropped {existing_cols_to_drop}")
    else:
        print(f"  {cat}: target column(s) not present, nothing to drop")

    write_parquet_safe(df, in_path)  # overwrite in place — same file, fewer columns

print("=" * 60)
print("Dropping info.winBuild from Memory data")
print("=" * 60)
for cat in CATEGORIES:
    drop_empty_columns(cat)

# %%
# ============================================================
# STEP 3: Normalize label inconsistencies
# ============================================================
# Map known label variants (e.g. "Zbenign" -> "Benign") so the Network and
# Memory sides — and all 9 categories — use one consistent label set.
# Skipping this creates a spurious 10th class after encoding.

def normalize_labels(prefix: str, cat: str) -> None:
    in_path = os.path.join(CLEAN_DIR, f"{prefix}_{cat}_dedup.parquet")
    if not os.path.exists(in_path):
        print(f"  {prefix}_{cat}: not found, skipping")
        return

    df = read_parquet_safe(in_path)
    if "label" not in df.columns:
        print(f"  {prefix}_{cat}: no 'label' column, skipping")
        return

    unique_labels_before = df["label"].unique().to_list()
    df = df.with_columns(
        pl.col("label").replace(LABEL_FIXES).alias("label")
    )
    unique_labels_after = df["label"].unique().to_list()

    if unique_labels_before != unique_labels_after:
        print(f"  {prefix}_{cat}: labels normalized {unique_labels_before} -> {unique_labels_after}")
    else:
        print(f"  {prefix}_{cat}: no label fixes needed ({unique_labels_after})")

    write_parquet_safe(df, in_path)

print("=" * 60)
print("Normalizing label inconsistencies (both sources)")
print("=" * 60)
for cat in CATEGORIES:
    normalize_labels("net", cat)
    normalize_labels("mem", cat)

# %%
# ============================================================
# STEP 4: Investigate missingness patterns BEFORE imputing
# ============================================================
# For delta_start / handshake_duration: check whether missingness is
# structural (concentrated in specific categories — a real behavioral
# signal) or random/logging noise, BEFORE choosing how to impute.
#
# Why this matters for bias: if missingness correlates strongly with one
# label, naive imputation lets the model (and SHAP) treat a data-capture
# artifact as if it were genuine malicious behavior.

MISSINGNESS_COLUMNS = ["delta_start", "handshake_duration"]

# Load all cleaned network categories together for this analysis —
# each category's file is now small enough post-dedup to do this safely.
net_frames = []
for cat in CATEGORIES:
    path = os.path.join(CLEAN_DIR, f"net_{cat}_dedup.parquet")
    if os.path.exists(path):
        net_frames.append(read_parquet_safe(path).with_columns(pl.lit(cat).alias("_source_cat")))

if net_frames:
    net_all = pl.concat(net_frames, how="diagonal_relaxed")

    print("=" * 60)
    print("Missingness rate per category (%) — for structural vs random check")
    print("=" * 60)
    for col in MISSINGNESS_COLUMNS:
        if col not in net_all.columns:
            print(f"  {col}: not found in network data")
            continue

        missing_by_cat = (
            net_all
            .group_by("_source_cat")
            .agg([
                pl.col(col).is_null().sum().alias("missing"),
                pl.len().alias("total"),
            ])
            .with_columns((pl.col("missing") / pl.col("total") * 100).alias("missing_pct"))
            .sort("missing_pct", descending=True)
        )
        print(f"\n--- {col} ---")
        print(missing_by_cat)

        pct_values = missing_by_cat["missing_pct"].to_list()
        spread = max(pct_values) - min(pct_values)

        # Heuristic: if missingness is heavily concentrated in specific
        # categories (big spread across categories), treat as structural.
        # If it's roughly uniform everywhere, treat as random/logging noise.
        if spread > 50:
            print(f"  -> DECISION: structural missingness (spread = {spread:.1f} pts). "
                  f"Fill with sentinel + add 'has_{col}' binary flag.")
        else:
            print(f"  -> DECISION: looks random/uniform (spread = {spread:.1f} pts). "
                  f"Use median imputation.")
else:
    print("No cleaned network files found yet — run Steps 1-3 first.")

# %%
# ============================================================
# STEP 4b: Apply the decision — sentinel + flag OR median imputation
# ============================================================
# EDIT the two lists below based on what the printed spread told you above.
STRUCTURAL_MISSING_COLUMNS = []  # e.g. ["delta_start", "handshake_duration"]
RANDOM_MISSING_COLUMNS = []      # e.g. leave empty if both are structural

def apply_missingness_strategy(cat: str) -> None:
    path = os.path.join(CLEAN_DIR, f"net_{cat}_dedup.parquet")
    if not os.path.exists(path):
        return
    df = read_parquet_safe(path)

    for col in STRUCTURAL_MISSING_COLUMNS:
        if col not in df.columns:
            continue
        flag_col = f"has_{col}"
        df = df.with_columns([
            pl.col(col).is_null().not_().cast(pl.Int8).alias(flag_col),
            pl.col(col).fill_null(-1).alias(col),
        ])

    for col in RANDOM_MISSING_COLUMNS:
        if col not in df.columns:
            continue
        median_val = df[col].median()
        df = df.with_columns(pl.col(col).fill_null(median_val).alias(col))

    write_parquet_safe(df, path)

if STRUCTURAL_MISSING_COLUMNS or RANDOM_MISSING_COLUMNS:
    print("Applying missingness strategy per category...")
    for cat in CATEGORIES:
        apply_missingness_strategy(cat)
    print("Done.")
else:
    print("STRUCTURAL_MISSING_COLUMNS / RANDOM_MISSING_COLUMNS are empty — "
          "fill these in based on the Step 4 output above, then re-run this cell.")

# %%
print("Steps 1-4 complete. Cleaned files are in:", CLEAN_DIR)

NETWORK — duplicate removal (compare against your known EDA counts)
  Backdoor: removed 10 duplicates (35122 -> 35112)
  Benign: removed 6 duplicates (67102 -> 67096)
  Exploit: removed 8 duplicates (47694 -> 47686)
  HackTool: removed 5 duplicates (61477 -> 61472)
  Hoax: removed 12 duplicates (54586 -> 54574)
  Rootkit: removed 8 duplicates (66857 -> 66849)
  Trojan: removed 14 duplicates (65880 -> 65866)
  Virus: removed 31 duplicates (179466 -> 179435)
  Worm: removed 6 duplicates (68491 -> 68485)
MEMORY — duplicate removal (not previously checked — new this run)
  Backdoor: removed 0 duplicates (1233 -> 1233)
  Benign: removed 0 duplicates (1125 -> 1125)
  Exploit: removed 0 duplicates (100 -> 100)
  HackTool: removed 5 duplicates (973 -> 968)
  Hoax: removed 0 duplicates (1099 -> 1099)
  Rootkit: removed 37 duplicates (1120 -> 1083)
  Trojan: removed 5 duplicates (1203 -> 1198)
  Virus: removed 398 duplicates (1608 -> 1210)
  Worm: removed 1 duplicates (1162 -> 1161)
Dropping inf